In [ ]:
%pip install -U google-generativeai
%pip install google-ai-generativelanguage==0.6.15
%pip install -U langchain-google-genai
%pip install -U langchain-community
%pip install -U langgraph

: 

In [12]:
import os 
import re 
import google.genai as genai 
from langgraph.graph import StateGraph, END 
from typing import TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

modelo = ChatOpenAI(
    model='gpt-5-nano',
    temperature=0.2,
    api_key=api_key
)

query = 'Afinal porque ola mundo?'

#print(modelo.invoke(query))


In [3]:
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system: 
            self.messages.append({"role":"system", "content":self.system})

    def __call__(self, message):
        self.messages.append({"role":"user", "content":message})
        result = self.execute()
        self.messages.append({"role":"assistant", "content":result})
        return result
    
    def execute(self):
        prompt = ""
        for msg in self.messages:
            prompt += f"{msg['role']}: {msg['content']}\n"

        response = modelo.invoke(prompt)
        return response.text
        
if __name__ == "__main__":
    agent = Agent(system="Você é um assistente útil e super hiper mega objetivo.")
    print(agent("qual eh a capital da franca?"))

A capital da França é Paris.


In [4]:
# consultor de estoque no inventario.
PROMPT_REACT = """
Você funciona em um ciclo de Pensamento, Ação, Pausa e Observação.
Ao final do ciclo, você fornece uma Resposta.
Use "Pensamento" para descrever seu raciocínio.
Use "Ação" para executar ferramentas - e então retorne "PAUSA".
A "Observação" será o resultado da ação executada.
Ações disponíveis:
  - consultar_estoque: retorna a quantidade disponível de um item no inventário (ex: "consultar_estoque: teclado")
  - consultar_preco_produto: retorna o preço unitário de um produto (ex: "consultar_preco_produto: mouse gamer")

Exemplo:
Pergunta: Quantos monitores temos em estoque?
Pensamento: Devo consultar a ação consultar_estoque para saber a quantidade de monitores.
Ação: consultar_estoque: monitor
PAUSA

Observação: Temos 75 monitores em estoque.
Resposta: Há 75 monitores em estoque.
""".strip()

In [5]:
#agent state

class AgentState(TypedDict):
    pergunta: str
    historico: list[str]
    acao_pendente: str
    resposta_final: str


In [6]:
def consultar_estoque(item: str) -> str:
    """
    Simula a consulta de estoque de um item no inventário.
    """
    item = item.lower()
    estoque = {
        "monitor": 75,
        "teclado": 120,
        "mouse gamer": 80,
        "webcam": 40,
        "headset": 60,
        "impressora": 15
    }

    if item in estoque:
        return f"Temos {estoque[item]} {item}s em estoque."
    else:
        return f"Item '{item}' não encontrado no inventário."

def consultar_preco_produto(produto: str) -> str:
    """
    Simula a consulta do preço unitário de um produto.
    """
    produto = produto.lower()
    precos = {
        "monitor": 999.90,
        "teclado": 150.00,
        "mouse gamer": 99.50,
        "webcam": 120.00,
        "headset": 180.00,
        "impressora": 750.00
    }

    if produto in precos:
        return f"O preço de um(a) {produto} é R$ {precos[produto]:.2f}."
    else:
        return f"Produto '{produto}' não encontrado na lista de preços."

In [8]:
print(consultar_estoque("teclado"))
print(consultar_preco_produto("impressora"))


Temos 120 teclados em estoque.
O preço de um(a) impressora é R$ 750.00.


In [ ]:
#gemini

def run_react_agent(pergunta: str, max_iterations: int = 5) -> str:
    model = genai.GenerativeModel('gemini-2.5-flash')
    chat = model.start_chat(history=[])

    chat.send_message(PROMPT_REACT)

    current_prompt = pergunta

    for i in range(max_iterations):
        response = chat.send_message(current_prompt)
        response_text = response.text.strip()

        print(f"--- Iteração {i+1} ---")
        print(f"Modelo pensou/respondeu:\n{response_text}\n")

        if response_text.startswith("Resposta:"):
            return response_text.replace("Resposta:", "").strip()

        match = re.search(r"Ação:\s*(\w+):\s*(.*)", response_text)

        if match:
            action_name = match.group(1).strip()
            action_arg = match.group(2).strip()

            observacao = ""
            if action_name == "consultar_estoque":
                observacao = consultar_estoque(action_arg)
            elif action_name == "consultar_preco_produto":
                observacao = consultar_preco_produto(action_arg)
            else:
                observacao = f"Erro: Ação '{action_name}' desconhecida."

            current_prompt = f"Observação: {observacao}\nResposta:"

            print(f"Executou ação: {action_name}({action_arg})")
            print(f"Observação: {observacao}\n")

        else:
            return f"Erro: O agente não conseguiu extrair uma Ação ou Resposta final após {i+1} iterações. Última resposta: {response_text}"

    return "Erro: Limite máximo de iterações atingido sem uma resposta final."

In [ ]:
#openAI

def run_react_agent(pergunta: str, max_iterations: int = 5) -> str:
    modelo = ChatOpenAI(
        model='gpt-5-nano', 
        temperature=0.2,
        api_key=api_key
    )
    
    mensagens = [
        SystemMessage(content=PROMPT_REACT),
        HumanMessage(content=pergunta)
    ]

    for i in range(max_iterations):
        response = modelo.invoke(mensagens)
        
        response_text = response.content.strip()

        print(f"--- Iteração {i+1} ---")
        print(f"Modelo pensou/respondeu:\n{response_text}\n")

        if response_text.startswith("Resposta:"):
            return response_text.replace("Resposta:", "").strip()

        match = re.search(r"Ação:\s*(\w+):\s*(.*)", response_text)

        if match:
            action_name = match.group(1).strip()
            action_arg = match.group(2).strip()

            observacao = ""
            if action_name == "consultar_estoque":
                observacao = consultar_estoque(action_arg)
            elif action_name == "consultar_preco_produto":
                observacao = consultar_preco_produto(action_arg)
            else:
                observacao = f"Erro: Ação '{action_name}' desconhecida."

            print(f"Executou ação: {action_name}({action_arg})")
            print(f"Observação: {observacao}\n")

            mensagens.append(AIMessage(content=response_text))
            mensagens.append(HumanMessage(content=f"Observação: {observacao}\nResposta:"))

        else:
            return f"Erro: O agente não conseguiu extrair uma Ação ou Resposta final após {i+1} iterações. Última resposta: {response_text}"

    return "Erro: Limite máximo de iterações atingido sem uma resposta final."

In [14]:
pergunta_1 = "Quantos mouses gamers estao no inventario?"
print(f"**Interacao 1: {pergunta_1}")
resposta_1 = run_react_agent(pergunta_1)
print(f"\n ** RESPOSTA FINAL DO AGENTE 1:* {resposta_1}\n")

print("\n" + "="*50 + "\n")

**Interacao 1: Quantos mouses gamers estao no inventario?
--- Iteração 1 ---
Modelo pensou/respondeu:
Pensamento: Preciso consultar a ação consultar_estoque para descobrir a quantidade de mouses gamers disponíveis no inventário.  
Ação: consultar_estoque: mouse gamer  
PAUSA

Executou ação: consultar_estoque(mouse gamer)
Observação: Temos 80 mouse gamers em estoque.

--- Iteração 2 ---
Modelo pensou/respondeu:
Há 80 mouses gamers em estoque.


 ** RESPOSTA FINAL DO AGENTE 1:* Erro: O agente não conseguiu extrair uma Ação ou Resposta final após 2 iterações. Última resposta: Há 80 mouses gamers em estoque.



